# 🌊 Marine Oil Spill Detection: SegNet vs. U-Net Comparison
### Train on Google Colab (with free GPU), evaluate, and compare both models

This self-contained notebook allows you to:
1. **Upload your dataset** (`dataset.zip` containing `train/` and `test/` splits for PALSAR and Sentinel-1).
2. **Define and Train SegNet and U-Net** on SAR satellite imagery.
3. **Compute all evaluation metrics**:
   - Accuracy, Precision, Recall, Specificity, F1 Score (Dice), IoU (Jaccard), ROC-AUC, Average Precision (PR-AUC).
4. **Compare SegNet vs. U-Net side-by-side**:
   - Comparison Metrics Table
   - Combined ROC Curves
   - Combined Precision-Recall Curves
   - Visual Inference Overlays (SAR Image vs Ground Truth vs SegNet vs U-Net)
5. **Download trained checkpoints & plots**.

---
> 💡 **Make sure GPU acceleration is enabled:**
> Go to **Runtime** → **Change runtime type** → select **T4 GPU** → **Save**.


In [ ]:
# Check GPU availability
import torch
import os

print("PyTorch version:", torch.__version__)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("⚠️ GPU not detected! Go to Runtime -> Change runtime type -> Select T4 GPU.")


In [ ]:
# Install any missing dependencies
!pip install -q matplotlib scikit-learn pandas pillow tqdm


## 📁 Step 1: Upload and Extract Dataset

Upload your `dataset.zip` directly to Colab. The expected zip structure:
```
dataset.zip
└── dataset/
    ├── train/
    │   ├── palsar/      (*_sat.jpg and *_mask.png in same folder)
    │   └── sentinel/    (*_sat.jpg and *_mask.png in same folder)
    └── test/
        ├── palsar/
        │   ├── sat/     (*_sat.jpg)
        │   └── gt/      (*_mask.png)
        └── sentinel/
            ├── sat/     (*_sat.jpg)
            └── gt/      (*_mask.png)
```
*You can upload `dataset.zip` using the left sidebar in Colab (Files icon) or use the cell below.*


In [ ]:
import os
import zipfile
from glob import glob

# --- OPTION B (Fastest): If you uploaded dataset.zip to your Google Drive, uncomment below: ---
# from google.colab import drive
# drive.mount('/content/drive')
# !cp "/content/drive/MyDrive/dataset.zip" .

zip_name = "dataset.zip"

if os.path.exists(zip_name):
    size_mb = os.path.getsize(zip_name) / (1024 * 1024)
    print(f"Found {zip_name} ({size_mb:.2f} MB)")
    
    if size_mb < 350:
        raise ValueError(
            f"❌ The uploaded file is only {size_mb:.2f} MB (expected ~372 MB)!\n"
            "The upload was either interrupted or is still uploading in the background.\n"
            "Please check the circular upload progress icon at the bottom-left of Colab and wait until it reaches 100%."
        )
    
    print(f"Extracting {zip_name}...")
    # Use Colab's fast shell unzip
    !unzip -q -o dataset.zip -d .
    print("✅ Extraction complete!")
elif os.path.exists("dataset"):
    print("✅ 'dataset' directory already exists!")
else:
    print(f"⚠️ {zip_name} not found! Please upload dataset.zip into Colab files.")

# Verification

train_palsar = glob("dataset/train/palsar/*_sat.jpg")
train_sentinel = glob("dataset/train/sentinel/*_sat.jpg")
test_palsar = glob("dataset/test/palsar/sat/*.*")
test_sentinel = glob("dataset/test/sentinel/sat/*.*")

print(f"Train samples - PALSAR: {len(train_palsar)}, Sentinel: {len(train_sentinel)}")
print(f"Test samples  - PALSAR: {len(test_palsar)}, Sentinel: {len(test_sentinel)}")


## 📦 Step 2: PyTorch Dataset Loader
Loads single-channel grayscale SAR images and binary ground truth masks, resized to 256×256.


In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms

class OilSpillDataset(Dataset):
    def __init__(self, split="train", sensor="palsar", base_dir="dataset"):
        self.split = split
        self.sensor = sensor
        self.base_dir = base_dir

        if split == "train":
            self.data_dir = os.path.join(base_dir, "train", sensor)
            if not os.path.exists(self.data_dir):
                self.image_paths, self.mask_paths = [], []
                return
            all_files = [f for f in os.listdir(self.data_dir) if f.endswith(".jpg")]
            self.image_paths = [os.path.join(self.data_dir, f) for f in all_files]
            
            self.mask_paths = []
            for img_path in self.image_paths:
                mask_name = os.path.basename(img_path).replace("_sat.jpg", "_mask.png")
                mask_path = os.path.join(self.data_dir, mask_name)
                self.mask_paths.append(mask_path if os.path.exists(mask_path) else None)
                
            valid_pairs = [(img, mask) for img, mask in zip(self.image_paths, self.mask_paths) if mask is not None]
            self.image_paths = [p[0] for p in valid_pairs]
            self.mask_paths  = [p[1] for p in valid_pairs]
        else:
            self.sat_dir = os.path.join(base_dir, "test", sensor, "sat")
            self.gt_dir  = os.path.join(base_dir, "test", sensor, "gt")
            if not os.path.exists(self.sat_dir):
                self.image_paths, self.mask_paths = [], []
                return
            self.image_paths = sorted([os.path.join(self.sat_dir, f) for f in os.listdir(self.sat_dir) if f.endswith((".jpg", ".png"))])
            self.mask_paths  = sorted([os.path.join(self.gt_dir,  f) for f in os.listdir(self.gt_dir)  if f.endswith((".jpg", ".png"))]) if os.path.exists(self.gt_dir) else None

        self.image_transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
        ])
        self.mask_transform = transforms.Compose([
            transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("L")
        img = self.image_transform(img)

        if self.mask_paths and idx < len(self.mask_paths) and self.mask_paths[idx] is not None:
            mask = Image.open(self.mask_paths[idx]).convert("L")
            mask = self.mask_transform(mask)
            mask = (mask > 0.5).float()
            return img, mask
        return img

def get_dataloaders(sensor="both", batch_size=8):
    sensors = ["palsar", "sentinel"] if sensor == "both" else [sensor]
    train_sets = [OilSpillDataset(split="train", sensor=s) for s in sensors]
    test_sets  = [OilSpillDataset(split="test",  sensor=s) for s in sensors]

    train_loader = DataLoader(ConcatDataset(train_sets), batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    test_loader  = DataLoader(ConcatDataset(test_sets),  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, test_loader

print("✅ OilSpillDataset class defined.")


## 🧠 Step 3: Model Architectures
- **SegNet**: Uses encoder max-pooling indices passed to decoder `max_unpool2d`.
- **U-Net**: Uses skip connections concatenating encoder feature maps directly into the decoder.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# ==========================================
# 1. SegNet
# ==========================================
class SegNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=1):
        super(SegNet, self).__init__()
        self.enc1 = self.conv_block(in_channels, 64)
        self.enc2 = self.conv_block(64, 128)
        self.enc3 = self.conv_block(128, 256)
        self.enc4 = self.conv_block(256, 512)
        self.enc5 = self.conv_block(512, 512)

        self.pool = nn.MaxPool2d(2, 2, return_indices=True)

        self.dec5 = self.conv_block(512, 512)
        self.dec4 = self.conv_block(512, 256)
        self.dec3 = self.conv_block(256, 128)
        self.dec2 = self.conv_block(128, 64)
        self.dec1 = nn.Conv2d(64, num_classes, kernel_size=1)

    def conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x1, idx1 = self.pool(self.enc1(x))
        x2, idx2 = self.pool(self.enc2(x1))
        x3, idx3 = self.pool(self.enc3(x2))
        x4, idx4 = self.pool(self.enc4(x3))
        x5, idx5 = self.pool(self.enc5(x4))

        x = F.max_unpool2d(x5, idx5, kernel_size=2)
        x = self.dec5(x)
        x = F.max_unpool2d(x, idx4, kernel_size=2)
        x = self.dec4(x)
        x = F.max_unpool2d(x, idx3, kernel_size=2)
        x = self.dec3(x)
        x = F.max_unpool2d(x, idx2, kernel_size=2)
        x = self.dec2(x)
        x = F.max_unpool2d(x, idx1, kernel_size=2)
        x = self.dec1(x)
        return x

# ==========================================
# 2. U-Net
# ==========================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=1, features=[64, 128, 256, 512]):
        super(UNet, self).__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        prev_channels = in_channels
        for feature in features:
            self.downs.append(DoubleConv(prev_channels, feature))
            prev_channels = feature

        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose2d(feature * 2, feature, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(feature * 2, feature))

        self.final_conv = nn.Conv2d(features[0], num_classes, kernel_size=1)

    def forward(self, x):
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip = skip_connections[idx // 2]
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=True)
            x = torch.cat((skip, x), dim=1)
            x = self.ups[idx + 1](x)

        return self.final_conv(x)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

m_segnet = SegNet().to(device)
m_unet   = UNet().to(device)
print(f"SegNet parameters : {count_parameters(m_segnet):,}")
print(f"U-Net  parameters : {count_parameters(m_unet):,}")


## 🚀 Step 4: Training Function
Standardized training loop tracking Loss, Accuracy, Precision, Recall, Specificity, F1 / Dice, and IoU with `ReduceLROnPlateau` and checkpoint saving.


In [ ]:
from tqdm.auto import tqdm
import csv

def compute_batch_metrics(preds, labels, smooth=1e-6):
    preds = (torch.sigmoid(preds) > 0.5).float().view(-1)
    labels = labels.float().view(-1)

    tp = (preds * labels).sum()
    fp = ((1 - labels) * preds).sum()
    fn = (labels * (1 - preds)).sum()
    tn = ((1 - labels) * (1 - preds)).sum()

    acc = (tp + tn) / (tp + tn + fp + fn + smooth)
    prec = tp / (tp + fp + smooth)
    rec = tp / (tp + fn + smooth)
    spec = tn / (tn + fp + smooth)
    f1 = 2 * prec * rec / (prec + rec + smooth)
    iou = tp / (tp + fp + fn + smooth)

    return {
        "accuracy": acc.item(), "precision": prec.item(), "recall": rec.item(),
        "specificity": spec.item(), "f1": f1.item(), "iou": iou.item()
    }

def train_model(model_name="unet", epochs=50, batch_size=8, lr=1e-3, sensor="both"):
    os.makedirs("models/saved_models", exist_ok=True)
    os.makedirs("outputs/training_plots", exist_ok=True)

    model = UNet(in_channels=1, num_classes=1).to(device) if model_name.lower() == "unet" else SegNet(in_channels=1, num_classes=1).to(device)
    
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, factor=0.5)

    train_loader, _ = get_dataloaders(sensor=sensor, batch_size=batch_size)
    print(f"\n--- Training {model_name.upper()} on {len(train_loader.dataset)} images for {epochs} epochs ---")

    history = {"epoch": [], "loss": [], "accuracy": [], "precision": [], "recall": [], "f1": [], "iou": [], "lr": []}
    best_loss = float("inf")
    best_model_path = f"models/saved_models/{model_name.lower()}_best_model.pth"

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, total_acc, total_prec, total_rec, total_f1, total_iou = 0, 0, 0, 0, 0, 0
        batches = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch:03d}/{epochs}", leave=False)
        for images, masks in pbar:
            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            m = compute_batch_metrics(outputs.detach(), masks)
            total_loss += loss.item()
            total_acc  += m["accuracy"]
            total_prec += m["precision"]
            total_rec  += m["recall"]
            total_f1   += m["f1"]
            total_iou  += m["iou"]
            batches += 1

            pbar.set_postfix({"loss": f"{loss.item():.4f}", "f1": f"{m['f1']:.4f}", "iou": f"{m['iou']:.4f}"})

        avg_loss = total_loss / batches
        avg_f1   = total_f1 / batches
        avg_iou  = total_iou / batches
        current_lr = optimizer.param_groups[0]["lr"]

        scheduler.step(avg_loss)

        history["epoch"].append(epoch)
        history["loss"].append(avg_loss)
        history["accuracy"].append(total_acc / batches)
        history["precision"].append(total_prec / batches)
        history["recall"].append(total_rec / batches)
        history["f1"].append(avg_f1)
        history["iou"].append(avg_iou)
        history["lr"].append(current_lr)

        print(f"Epoch {epoch:03d} | Loss: {avg_loss:.4f} | F1: {avg_f1:.4f} | IoU: {avg_iou:.4f} | LR: {current_lr:.6f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), best_model_path)
            print(f"  ⭐ Saved new best model to {best_model_path}")

    # Save final model & history CSV
    final_path = f"models/saved_models/{model_name.lower()}_final_model.pth"
    torch.save(model.state_dict(), final_path)
    csv_path = f"outputs/training_plots/{model_name.lower()}_history.csv"
    
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=history.keys())
        writer.writeheader()
        for i in range(len(history["epoch"])):
            writer.writerow({k: history[k][i] for k in history})

    print(f"✅ Training completed for {model_name.upper()}!")
    return model, history, best_model_path


## 🏋️‍♂️ Step 5: Train Models
Run the cells below to train **U-Net** (and optionally **SegNet**).
*(You can adjust `epochs` and `batch_size` based on your training time preferences, e.g. 30-50 epochs on Colab T4 GPU takes ~10-20 minutes).*


In [ ]:
# Train U-Net
unet_model, unet_history, unet_best_path = train_model(
    model_name="unet",
    epochs=50,       # Change to 100 for full convergence
    batch_size=8,    # 8 fits comfortably in Colab 16GB GPU
    lr=1e-3,
    sensor="both"
)


In [ ]:
# (Optional) Train SegNet if you also want to retrain SegNet from scratch on Colab
# If you already have segnet_best_model.pth, you can just upload it to models/saved_models/

# segnet_model, segnet_history, segnet_best_path = train_model(
#     model_name="segnet",
#     epochs=50,
#     batch_size=8,
#     lr=1e-3,
#     sensor="both"
# )


## 📊 Step 6: Full Evaluation on Test Set
Computes all pixel-level metrics across the unseen test set (Accuracy, Precision, Recall, Specificity, F1 / Dice, IoU, ROC-AUC, Average Precision).


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score
)

def evaluate_model(model_class, checkpoint_path, sensor="both"):
    model = model_class(in_channels=1, num_classes=1).to(device)
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()

    resize = transforms.Compose([transforms.Resize((256, 256)), transforms.ToTensor()])
    resize_mask = transforms.Compose([transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST), transforms.ToTensor()])

    sensors = ["palsar", "sentinel"] if sensor == "both" else [sensor]
    all_probs, all_labels = [], []

    print(f"Evaluating {checkpoint_path}...")
    for s in sensors:
        sat_dir = os.path.join("dataset", "test", s, "sat")
        gt_dir  = os.path.join("dataset", "test", s, "gt")
        image_paths = sorted([os.path.join(sat_dir, f) for f in os.listdir(sat_dir) if f.endswith((".jpg", ".png"))])
        mask_paths  = sorted([os.path.join(gt_dir,  f) for f in os.listdir(gt_dir)  if f.endswith((".jpg", ".png"))])

        for img_path, mask_path in zip(image_paths, mask_paths):
            img  = resize(Image.open(img_path).convert("L")).unsqueeze(0).to(device)
            mask = resize_mask(Image.open(mask_path).convert("L"))

            with torch.no_grad():
                prob = torch.sigmoid(model(img)).squeeze().cpu().numpy().flatten()

            all_probs.append(prob)
            all_labels.append((mask.numpy().flatten() > 0.5).astype(np.uint8))

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    all_preds  = (all_probs > 0.5).astype(np.uint8)

    smooth = 1e-7
    tp = int(((all_preds == 1) & (all_labels == 1)).sum())
    fp = int(((all_preds == 1) & (all_labels == 0)).sum())
    fn = int(((all_preds == 0) & (all_labels == 1)).sum())
    tn = int(((all_preds == 0) & (all_labels == 0)).sum())

    acc  = (tp + tn) / (tp + tn + fp + fn + smooth)
    prec = tp / (tp + fp + smooth)
    rec  = tp / (tp + fn + smooth)
    spec = tn / (tn + fp + smooth)
    f1   = 2 * prec * rec / (prec + rec + smooth)
    iou  = tp / (tp + fp + fn + smooth)
    dice = 2 * tp / (2 * tp + fp + fn + smooth)

    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    roc_auc = float(auc(fpr, tpr))
    prec_c, rec_c, _ = precision_recall_curve(all_labels, all_probs)
    ap = float(average_precision_score(all_labels, all_probs))

    metrics = {
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": spec,
        "F1 / Dice": f1,
        "IoU": iou,
        "ROC-AUC": roc_auc,
        "Average Precision": ap,
        "TP": tp, "FP": fp, "FN": fn, "TN": tn
    }

    curves = {
        "fpr": fpr, "tpr": tpr, "roc_auc": roc_auc,
        "precision_curve": prec_c, "recall_curve": rec_c, "ap": ap
    }
    return metrics, curves


In [ ]:
# Run evaluation on UNet
unet_metrics, unet_curves = evaluate_model(UNet, "models/saved_models/unet_best_model.pth")

# If you have segnet_best_model.pth in models/saved_models/:
segnet_path = "models/saved_models/segnet_best_model.pth"
if os.path.exists(segnet_path):
    segnet_metrics, segnet_curves = evaluate_model(SegNet, segnet_path)
else:
    # Use baseline SegNet results from repository
    segnet_metrics = {
        "Accuracy": 0.886544, "Precision": 0.813123, "Recall": 0.734259,
        "Specificity": 0.940362, "F1 / Dice": 0.771681, "IoU": 0.628242,
        "ROC-AUC": 0.940979, "Average Precision": 0.860310,
        "TP": 20293019, "FP": 4663866, "FN": 7344389, "TN": 73539366
    }
    segnet_curves = None

# Create Side-by-Side Comparison Table
df_compare = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "Specificity", "F1 / Dice", "IoU", "ROC-AUC", "Average Precision"],
    "SegNet": [f"{segnet_metrics[m]*100:.2f}%" for m in ["Accuracy", "Precision", "Recall", "Specificity", "F1 / Dice", "IoU", "ROC-AUC", "Average Precision"]],
    "U-Net":  [f"{unet_metrics[m]*100:.2f}%" for m in ["Accuracy", "Precision", "Recall", "Specificity", "F1 / Dice", "IoU", "ROC-AUC", "Average Precision"]]
})

print("\n" + "="*50)
print("        SEGNET vs. U-NET COMPARISON TABLE")
print("="*50)
display(df_compare)


## 📈 Step 7: Comparative ROC & Precision-Recall Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1. ROC Curves
ax = axes[0]
if segnet_curves:
    ax.plot(segnet_curves["fpr"], segnet_curves["tpr"], color="steelblue", lw=2, label=f"SegNet (AUC = {segnet_curves['roc_auc']:.4f})")
else:
    ax.plot([], [], ' ', label=f"SegNet (AUC = {segnet_metrics['ROC-AUC']:.4f})")
ax.plot(unet_curves["fpr"], unet_curves["tpr"], color="crimson", lw=2, label=f"U-Net (AUC = {unet_curves['roc_auc']:.4f})")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", lw=1)
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curve: SegNet vs. U-Net", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=11)
ax.grid(True, alpha=0.3)

# 2. Precision-Recall Curves
ax = axes[1]
if segnet_curves:
    ax.plot(segnet_curves["recall_curve"], segnet_curves["precision_curve"], color="steelblue", lw=2, label=f"SegNet (AP = {segnet_curves['ap']:.4f})")
else:
    ax.plot([], [], ' ', label=f"SegNet (AP = {segnet_metrics['Average Precision']:.4f})")
ax.plot(unet_curves["recall_curve"], unet_curves["precision_curve"], color="crimson", lw=2, label=f"U-Net (AP = {unet_curves['ap']:.4f})")
ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Precision-Recall Curve: SegNet vs. U-Net", fontsize=14, fontweight="bold")
ax.legend(loc="lower left", fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs("outputs/evaluation", exist_ok=True)
plt.savefig("outputs/evaluation/segnet_vs_unet_curves.png", dpi=200)
plt.show()


## 🔍 Step 8: Visual Segmentation Predictions
Side-by-side comparison on sample test images:
$$\text{Original SAR Image} \quad | \quad \text{Ground Truth Mask} \quad | \quad \text{SegNet Prediction} \quad | \quad \text{U-Net Prediction}$$


In [ ]:
import cv2
import glob
import random

def predict_mask(model, img_tensor):
    model.eval()
    with torch.no_grad():
        prob = torch.sigmoid(model(img_tensor.unsqueeze(0).to(device)))
        pred = (prob > 0.5).squeeze().cpu().numpy().astype(np.uint8)
    return pred

# Select test images
test_imgs = sorted(glob.glob("dataset/test/palsar/sat/*_sat.jpg") + glob.glob("dataset/test/sentinel/sat/*_sat.jpg"))
selected = random.sample(test_imgs, min(4, len(test_imgs)))

resize = transforms.Compose([transforms.Resize((256, 256)), transforms.ToTensor()])
resize_mask = transforms.Compose([transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST), transforms.ToTensor()])

unet = UNet().to(device)
unet.load_state_dict(torch.load("models/saved_models/unet_best_model.pth", map_location=device))

has_segnet = os.path.exists("models/saved_models/segnet_best_model.pth")
if has_segnet:
    segnet = SegNet().to(device)
    segnet.load_state_dict(torch.load("models/saved_models/segnet_best_model.pth", map_location=device))

fig, axes = plt.subplots(len(selected), 4 if has_segnet else 3, figsize=(16, 4 * len(selected)))
if len(selected) == 1:
    axes = np.expand_dims(axes, 0)

for i, img_path in enumerate(selected):
    # Load input
    raw_pil = Image.open(img_path).convert("L")
    img_tensor = resize(raw_pil)
    sar_np = np.array(raw_pil.resize((256, 256)))

    # Load GT mask
    gt_path = img_path.replace("/sat/", "/gt/").replace("_sat.jpg", "_mask.png")
    gt_np = np.array(Image.open(gt_path).convert("L").resize((256, 256), Image.NEAREST)) if os.path.exists(gt_path) else np.zeros((256, 256))

    # U-Net prediction
    unet_pred = predict_mask(unet, img_tensor)

    # Plot
    col = 0
    axes[i, col].imshow(sar_np, cmap="gray")
    axes[i, col].set_title("Original SAR Image", fontweight="bold"); axes[i, col].axis("off"); col += 1

    axes[i, col].imshow(gt_np, cmap="gray")
    axes[i, col].set_title("Ground Truth Mask", fontweight="bold"); axes[i, col].axis("off"); col += 1

    if has_segnet:
        segnet_pred = predict_mask(segnet, img_tensor)
        axes[i, col].imshow(segnet_pred, cmap="inferno")
        axes[i, col].set_title("SegNet Prediction", fontweight="bold"); axes[i, col].axis("off"); col += 1

    axes[i, col].imshow(unet_pred, cmap="viridis")
    axes[i, col].set_title("U-Net Prediction", fontweight="bold"); axes[i, col].axis("off")

plt.tight_layout()
plt.savefig("outputs/evaluation/visual_comparison.png", dpi=200)
plt.show()


## 💾 Step 9: Download Trained Models & Results
Zip up the generated weights, metrics, and plots for easy download to your local machine.


In [ ]:
# Zip outputs and checkpoints
!zip -r unet_results_and_models.zip models/ outputs/

from google.colab import files
files.download("unet_results_and_models.zip")
print("✅ Downloading unet_results_and_models.zip...")
